# TAHAP 1 — Generate Synthetic Fraud Data

Membuat **dataset sintetis** transaksi UMKM untuk melatih model deteksi fraud refund.

**Output:** `data/raw/synthetic_transactions.csv`

**Label Severity (multi-class, bukan binary 0/1):**
- **LOW** — Transaksi normal, tidak ada indikasi fraud
- **MEDIUM** — Profil C: refund nominal kecil/wajar tapi frekuensi abnormal
- **HIGH** — Profil B: refund nominal menengah, berulang di jam normal
- **CRITICAL** — Profil A: refund besar, jam dini hari, beruntun cepat

**Penting:** `fraud_severity` adalah "kunci jawaban" untuk evaluasi. Model TIDAK PERNAH melihat kolom ini saat training.

## Import & Konfigurasi

In [1]:
import os
import uuid
import numpy as np
import pandas as pd
import sqlite3
from datetime import datetime, timedelta

# Konfigurasi
N_NORMAL = 2500                              # jumlah transaksi normal
CASHIERS = [f"CSH-{i:03d}" for i in range(1, 6)]   # 5 kasir
RNG_SEED = 42

NORMAL_HOURS = (8, 21)                       # jam operasional toko
SALE_MIN, SALE_MAX = 10_000, 250_000        # nominal jual wajar UMKM (Rp)

rng = np.random.default_rng(RNG_SEED)       # satu RNG global, reproducible
BASE_TIME = datetime(2026, 5, 1, 8, 0, 0)

# Output paths (relatif dari folder notebooks/)
OUTPUT_DIR = os.path.join("..", "data", "raw")
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "synthetic_transactions.csv")

print("[OK] Konfigurasi:")
print(f"     Jumlah normal: {N_NORMAL:,}")
print(f"     Kasir: {', '.join(CASHIERS)}")
print(f"     Output: {OUTPUT_CSV}")

[OK] Konfigurasi:
     Jumlah normal: 2,500
     Kasir: CSH-001, CSH-002, CSH-003, CSH-004, CSH-005
     Output: ..\data\raw\synthetic_transactions.csv


## Fungsi Pembantu (Vectorized)

In [2]:
def gen_uuids(n: int) -> np.ndarray:
    """Generate N UUID v4 sekaligus menggunakan numpy random bytes (batch, cepat)."""
    raw = rng.integers(0, 256, size=(n, 16), dtype=np.uint8)
    # Set version=4 dan variant bits sesuai RFC 4122
    raw[:, 6] = (raw[:, 6] & 0x0F) | 0x40
    raw[:, 8] = (raw[:, 8] & 0x3F) | 0x80
    return np.array([str(uuid.UUID(bytes=bytes(row))) for row in raw])


def gen_timestamps_vectorized(n: int, day_lo: int, day_hi: int,
                               hour_lo: int, hour_hi: int) -> np.ndarray:
    """Generate N timestamps vectorized — jauh lebih cepat dari loop."""
    days    = rng.integers(day_lo, day_hi,  size=n)
    hours   = rng.integers(hour_lo, hour_hi, size=n)
    minutes = rng.integers(0, 60, size=n)
    seconds = rng.integers(0, 60, size=n)
    # Vectorized: BASE_TIME + offset dalam detik
    offsets = (days.astype('int64') * 86_400
               + hours.astype('int64') * 3_600
               + minutes.astype('int64') * 60
               + seconds.astype('int64'))
    base_ts = np.datetime64(BASE_TIME)
    return base_ts + offsets.astype('timedelta64[s]')


print("[OK] Fungsi pembantu (vectorized) siap")

[OK] Fungsi pembantu (vectorized) siap


## Generate Transaksi Normal (LOW)

In [3]:
print("[1/5] Generate transaksi NORMAL (severity: LOW)...")

n = N_NORMAL
cashiers_normal = rng.choice(CASHIERS, size=n)
types_normal    = rng.choice(["SALE", "REFUND"], size=n, p=[0.92, 0.08])
amounts_normal  = np.round(rng.uniform(SALE_MIN, SALE_MAX, size=n), 2)
amounts_normal += rng.normal(0, 5000, size=n)
amounts_normal  = np.clip(amounts_normal, 1000, None)
amounts_normal  = np.round(amounts_normal, 2)

ts_normal       = gen_timestamps_vectorized(n, 0, 30, NORMAL_HOURS[0], NORMAL_HOURS[1])
ids_normal      = gen_uuids(n)

df_normal = pd.DataFrame({
    "id":               ids_normal,
    "cashier_id":       cashiers_normal,
    "timestamp":        ts_normal.astype('datetime64[ms]').astype(str),
    "transaction_type": types_normal,
    "amount":           amounts_normal,
    "is_synced":        np.ones(n, dtype=np.int8),
    "fraud_severity":   "LOW",
    "fraud_profile":    "normal",
})
n_sale   = (df_normal["transaction_type"] == "SALE").sum()
n_refund = (df_normal["transaction_type"] == "REFUND").sum()
print(f"      {len(df_normal):,} transaksi normal dibuat")
print(f"      Breakdown: SALE {n_sale:,}, REFUND {n_refund:,}")

[1/5] Generate transaksi NORMAL (severity: LOW)...
      2,500 transaksi normal dibuat
      Breakdown: SALE 2,284, REFUND 216


## Generate Fraud Profil A — CRITICAL

Kasir melakukan refund nominal besar (Rp 500K–2M) secara beruntun cepat di **jam dini hari** (di luar jam operasional). Pola yang sangat mencolok.

In [4]:
print("[2/5] Generate FRAUD Profil A — CRITICAL")

fraud_cashier_a = CASHIERS[0]
N_A = 50

amounts_a   = np.round(rng.uniform(300_000, 2_000_000, size=N_A), 2)
amounts_a  += rng.normal(0, 25000, size=N_A)
amounts_a   = np.round(amounts_a, 2)

intervals_a = rng.integers(15, 60, size=N_A).astype('int64')
intervals_a = np.clip(intervals_a + rng.integers(-5, 5, size=N_A), 5, 120)
cumulative_a = np.cumsum(intervals_a)
base_a = np.datetime64(BASE_TIME + timedelta(days=26, hours=1))
ts_a = base_a + cumulative_a.astype('timedelta64[s]')

df_a = pd.DataFrame({
    "id":               gen_uuids(N_A),
    "cashier_id":       fraud_cashier_a,
    "timestamp":        ts_a.astype('datetime64[ms]').astype(str),
    "transaction_type": "REFUND",
    "amount":           amounts_a,
    "is_synced":        np.ones(N_A, dtype=np.int8),
    "fraud_severity":   "CRITICAL",
    "fraud_profile":    "A",
})
print(f"      Kasir: {fraud_cashier_a}")
print(f"      Jumlah: {N_A} refund besar (Rp 300K-2M)")
print(f"      Waktu: jam dini hari, beruntun cepat")

[2/5] Generate FRAUD Profil A — CRITICAL
      Kasir: CSH-001
      Jumlah: 50 refund besar (Rp 300K-2M)
      Waktu: jam dini hari, beruntun cepat


## Generate Fraud Profil B — HIGH

Refund nominal menengah (Rp 200K–400K) dilakukan oleh **2 kasir** di jam kerja normal. Lebih tersamar karena waktunya wajar, tapi pola refund berulang.

In [5]:
print("[3/5] Generate FRAUD Profil B — HIGH")

N_B_PER = 60
N_B = N_B_PER * 2

cashiers_b = np.repeat([CASHIERS[2], CASHIERS[3]], N_B_PER)
amounts_b  = np.round(rng.uniform(100_000, 400_000, size=N_B), 2)
amounts_b += rng.normal(0, 15000, size=N_B)
amounts_b  = np.round(amounts_b, 2)

ts_b       = gen_timestamps_vectorized(N_B, 20, 30, 10, 18)

df_b = pd.DataFrame({
    "id":               gen_uuids(N_B),
    "cashier_id":       cashiers_b,
    "timestamp":        ts_b.astype('datetime64[ms]').astype(str),
    "transaction_type": "REFUND",
    "amount":           amounts_b,
    "is_synced":        np.ones(N_B, dtype=np.int8),
    "fraud_severity":   "HIGH",
    "fraud_profile":    "B",
})
print(f"      Kasir: {CASHIERS[2]}, {CASHIERS[3]}")
print(f"      Jumlah: {N_B} refund (Rp 100K-400K)")
print(f"      Waktu: jam kerja normal (10-18)")

[3/5] Generate FRAUD Profil B — HIGH
      Kasir: CSH-003, CSH-004
      Jumlah: 120 refund (Rp 100K-400K)
      Waktu: jam kerja normal (10-18)


## Generate Fraud Profil C — MEDIUM

Refund nominal kecil/wajar (Rp 40K–120K) tapi dengan **frekuensi abnormal** — 12 refund per hari selama 6 hari berturut-turut. Tipe fraud yang paling halus dan sulit dideteksi.

In [6]:
print("[4/5] Generate FRAUD Profil C — MEDIUM")

DAYS_C = [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29] # 15 hari
N_C_PER_DAY = 12
N_C = len(DAYS_C) * N_C_PER_DAY     # 180 total
fraud_cashier_c = CASHIERS[4]

amounts_c = np.round(rng.uniform(40_000, 120_000, size=N_C), 2)
amounts_c += rng.normal(0, 5000, size=N_C)
amounts_c = np.round(amounts_c, 2)

ts_c_list = []
for day in DAYS_C:
    start_hour = int(rng.integers(9, 17))
    intervals  = rng.integers(2, 12, size=N_C_PER_DAY).astype('int64') * 60
    intervals = np.clip(intervals + rng.integers(-30, 30, size=N_C_PER_DAY), 60, 3600)
    base_day   = np.datetime64(BASE_TIME + timedelta(days=day, hours=start_hour))
    ts_c_list.append(base_day + np.cumsum(intervals).astype('timedelta64[s]'))
ts_c = np.concatenate(ts_c_list)

df_c = pd.DataFrame({
    "id":               gen_uuids(N_C),
    "cashier_id":       fraud_cashier_c,
    "timestamp":        ts_c.astype('datetime64[ms]').astype(str),
    "transaction_type": "REFUND",
    "amount":           amounts_c,
    "is_synced":        np.ones(N_C, dtype=np.int8),
    "fraud_severity":   "MEDIUM",
    "fraud_profile":    "C",
})
print(f"      Kasir: {fraud_cashier_c}")
print(f"      Jumlah: {N_C} refund kecil frekuensi abnormal (Rp 40K-120K)")
print(f"      Waktu: {len(DAYS_C)} hari x {N_C_PER_DAY} refund/hari")

[4/5] Generate FRAUD Profil C — MEDIUM
      Kasir: CSH-005
      Jumlah: 180 refund kecil frekuensi abnormal (Rp 40K-120K)
      Waktu: 15 hari x 12 refund/hari


## Generate Fraud Profil D — CRITICAL (Kombinasi Sinyal SALE)

Kombinasi anomali: SALE, larut malam, nominal tinggi, frekuensi tinggi, kasir tertentu (CSH-002).

In [7]:
print("[5/5] Generate FRAUD Profil D — CRITICAL (SALE Kombinasi)")
N_D = 50 # Total CRITICAL = 50 (Profil A) + 50 (Profil D) = 100
amounts_d = np.round(rng.uniform(150_000, 350_000, size=N_D), 2)
amounts_d += rng.normal(0, 10000, size=N_D)
amounts_d = np.clip(np.round(amounts_d, 2), 5000, None)

# Randomisasi Kasir untuk Profil D (Mencegah memorization pada satu cashier)
cashiers_d = rng.choice(CASHIERS, size=N_D)

ts_d_list = []
DAYS_D = [5, 9, 12, 15, 19] # 5 hari
N_D_PER_DAY = N_D // len(DAYS_D) # 10 per hari
for day in DAYS_D:
    start_hour = int(rng.choice([23, 0, 1]))
    intervals  = rng.integers(15, 45, size=N_D_PER_DAY).astype('int64')
    intervals = np.clip(intervals + rng.integers(-5, 5, size=N_D_PER_DAY), 5, 120)
    base_day   = np.datetime64(BASE_TIME + timedelta(days=day, hours=start_hour))
    ts_d_list.append(base_day + np.cumsum(intervals).astype('timedelta64[s]'))
ts_d = np.concatenate(ts_d_list)

df_d = pd.DataFrame({
    "id":               gen_uuids(N_D),
    "cashier_id":       cashiers_d,
    "timestamp":        ts_d.astype('datetime64[ms]').astype(str),
    "transaction_type": "SALE",
    "amount":           amounts_d,
    "is_synced":        np.ones(N_D, dtype=np.int8),
    "fraud_severity":   "CRITICAL",
    "fraud_profile":    "D",
})
print(f"      Kasir: [Random]")
print(f"      Jumlah: {N_D} SALE anomali malam (Rp 150K-350K)")

[5/5] Generate FRAUD Profil D — CRITICAL (SALE Kombinasi)
      Kasir: [Random]
      Jumlah: 50 SALE anomali malam (Rp 150K-350K)


## Gabungkan & Simpan ke CSV + SQLite

In [8]:
print("Menggabungkan & menyimpan...")

# Gabungkan semua DataFrame
df = pd.concat([df_normal, df_a, df_b, df_c, df_d], ignore_index=True)

# Parse timestamp & sort kronologis
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Tipe data optimal
df["amount"]   = df["amount"].astype("float64")
df["is_synced"] = df["is_synced"].astype("int8")

# Simpan ke CSV (output utama untuk ML pipeline)
os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f"  CSV disimpan: {OUTPUT_CSV}")
print(f"  Total baris: {len(df):,}")

# Juga simpan ke SQLite agar API bisa langsung dipakai
import sqlite3
DB_DIR  = os.path.join("..", "database")
DB_PATH = os.path.join(DB_DIR, "local_pos.db")
os.makedirs(DB_DIR, exist_ok=True)

df_db = df.copy()
df_db["is_fraud"] = (df_db["fraud_severity"] != "LOW").astype("int8")

conn = sqlite3.connect(DB_PATH)
df_db.to_sql("transactions", conn, if_exists="replace", index=False,
             method="multi", chunksize=500)
conn.close()
print(f"  SQLite disimpan: {DB_PATH}")

Menggabungkan & menyimpan...
  CSV disimpan: ..\data\raw\synthetic_transactions.csv
  Total baris: 2,900
  SQLite disimpan: ..\database\local_pos.db


## Ringkasan & Distribusi Severity

In [9]:
# Ringkasan data
total = len(df)
sev_counts = df["fraud_severity"].value_counts()

print("RINGKASAN DATA")
print(f"\n  Total transaksi: {total:,}")
print(f"\n  Distribusi Severity:")
for sev in ["LOW", "MEDIUM", "HIGH", "CRITICAL"]:
    cnt = sev_counts.get(sev, 0)
    pct = cnt / total * 100
    print(f"    {sev:10s}: {cnt:5,} ({pct:.1f}%)")

print(f"\n  Sebaran fraud per kasir:")
fraud_only = df[df["fraud_severity"] != "LOW"]
fraud_dist = fraud_only.groupby("cashier_id")["fraud_severity"].value_counts()
for (cashier, sev), count in fraud_dist.items():
    print(f"    {cashier}: {count:3d} x {sev}")

print(f"\n  Jenis transaksi:")
print(f"    SALE  : {(df['transaction_type'] == 'SALE').sum():,}")
print(f"    REFUND: {(df['transaction_type'] == 'REFUND').sum():,}")

print(f"\n  Nominal (Rp):")
print(f"    Min   : {df['amount'].min():,.0f}")
print(f"    Max   : {df['amount'].max():,.0f}")
print(f"    Mean  : {df['amount'].mean():,.0f}")
print(f"    Median: {df['amount'].median():,.0f}")

print("\nData siap untuk Tahap 2 (EDA & Preprocessing)")

RINGKASAN DATA

  Total transaksi: 2,900

  Distribusi Severity:
    LOW       : 2,500 (86.2%)
    MEDIUM    :   180 (6.2%)
    HIGH      :   120 (4.1%)
    CRITICAL  :   100 (3.4%)

  Sebaran fraud per kasir:
    CSH-001:  62 x CRITICAL
    CSH-002:  10 x CRITICAL
    CSH-003:  60 x HIGH
    CSH-003:   5 x CRITICAL
    CSH-004:  60 x HIGH
    CSH-004:  13 x CRITICAL
    CSH-005: 180 x MEDIUM
    CSH-005:  10 x CRITICAL

  Jenis transaksi:
    SALE  : 2,334
    REFUND: 566

  Nominal (Rp):
    Min   : 2,827
    Max   : 1,901,334
    Mean  : 150,062
    Median: 126,460

Data siap untuk Tahap 2 (EDA & Preprocessing)
